In [114]:
import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


## Осмотр и обработка данных

In [115]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 11091 entries, 0 to 11090
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   cookie_id          11091 non-null  str           
 1   cookie_created_at  11091 non-null  datetime64[us]
 2   window_start_ts    11091 non-null  datetime64[us]
 3   window_end_ts      11091 non-null  datetime64[us]
 4   target             11091 non-null  int64         
dtypes: datetime64[us](3), int64(1), str(1)
memory usage: 639.2 KB


In [116]:
print(train.duplicated().sum())
train.isna().sum()

0


cookie_id            0
cookie_created_at    0
window_start_ts      0
window_end_ts        0
target               0
dtype: int64

In [117]:
print(test.duplicated().sum())
test.isna().sum()

0


cookie_id            0
cookie_created_at    0
window_start_ts      0
window_end_ts        0
dtype: int64

In [118]:
print(events.duplicated().sum())
events.isna().sum()

4863


cookie_id             0
event_ts              0
eid                   0
event_name            0
platform              0
user_agent            0
item_id          114597
item_category     32920
item_location     23793
seller_type      133853
search_query     228503
search_page      228503
pointer_x        220365
pointer_y        220365
dtype: int64

In [119]:
events.loc[events.duplicated(keep=False)].sort_values(["cookie_id", "event_ts"])

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
37283,ck_0027b5d4107d65d8,2026-04-26 05:22:48,300,contact_phone_show,android,Mozilla/5.0 (Linux; Android 14; Redmi Note 11)...,1043647.0,elektronika,krasnoyarsk,private,NaN,NaN,NaN,NaN
141860,ck_0027b5d4107d65d8,2026-04-26 05:22:48,300,contact_phone_show,android,Mozilla/5.0 (Linux; Android 14; Redmi Note 11)...,1043647.0,elektronika,krasnoyarsk,private,NaN,NaN,NaN,NaN
314150,ck_0027ba644f9ae2cd,2026-04-22 17:06:55,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,1036126.0,kvartiry_prodazha,samara,pro,NaN,NaN,NaN,NaN
321075,ck_0027ba644f9ae2cd,2026-04-22 17:06:55,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,1036126.0,kvartiry_prodazha,samara,pro,NaN,NaN,NaN,NaN
10173,ck_0032c08d8987d0ad,2026-04-10 18:13:05,210,photo_swipe,android,Mozilla/5.0 (Linux; Android 14; SM-G991B) Appl...,1051179.0,NaN,moskva,pro,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182287,ck_fff9fdc2adf5100b,2026-04-23 21:32:38,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1004624.0,elektronika,sankt-peterburg,pro,NaN,NaN,795.0,946.0
148169,ck_fffa4b3ab0203fcf,2026-04-19 16:35:52,100,search_results_view,Android,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,NaN,zhivotnye,ekaterinburg,NaN,котята,3.0,NaN,NaN
328579,ck_fffa4b3ab0203fcf,2026-04-19 16:35:52,100,search_results_view,Android,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,NaN,zhivotnye,ekaterinburg,NaN,котята,3.0,NaN,NaN
91965,ck_fffa4b3ab0203fcf,2026-04-19 22:11:32,200,item_view,ANDROID,Mozilla/5.0 (Linux; Android 11; Pixel 6) Apple...,1043761.0,zhivotnye,sankt-peterburg,pro,NaN,NaN,NaN,NaN


Пока дубли не буду никак обрабатывать, т.к. они это необязательно ошибки, несколько событий могли произойти в одну секунду, что в целом может говорить о действиях бота

Пропуски пока тоже оставлю, они наоборот дают полезную информацию. Например, пропуск в pointer_x означает, что для события координата не записывалась, пропуск в search_query нормален для просмотра объявления

In [120]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 



In [121]:
events.platform = events.platform.str.lower().str.strip()
events.platform = events.platform.replace({"iphone": "ios"})

In [122]:
print(events.platform.value_counts(), '\n')

platform
web        138792
android    126875
desktop     46866
ios         16372
Name: count, dtype: int64 



In [123]:
train['cookie_id'].isin(test['cookie_id']).sum()

np.int64(0)

На всякий случай проверил нет ли пересечений

In [124]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


In [125]:
train.target.unique()

array([0, 1])

## Создание признаков

Начал с тех признаков, которые основаны на времени 

In [126]:
ev_tr.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
0,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0,2026-04-13,2026-04-14
1,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN,2026-04-08,2026-04-09
2,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN,2026-04-12,2026-04-13
4,ck_f80d85f038cdacee,2026-04-11 16:08:12,200,item_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1000845.0,bytovaya_tehnika,sankt-peterburg,private,NaN,NaN,NaN,NaN,2026-04-11,2026-04-12
5,ck_e4f046f721d10f81,2026-04-10 16:50:05,200,item_view,ios,Mozilla/5.0 (iPhone; CPU iPhone OS 15_2 like M...,1055188.0,telefony,ekaterinburg,pro,NaN,NaN,NaN,NaN,2026-04-10,2026-04-11


In [131]:
actions = ev_tr.groupby('cookie_id').agg(
        n_events=('event_ts', 'size'),
        first_event=('event_ts', 'min'),
        last_event=('event_ts', 'max')).reset_index()
actions.head()

,cookie_id,n_events,first_event,last_event
0,ck_000c95f1408dcb00,16,2026-04-19 14:49:54,2026-04-19 23:11:51
1,ck_000e8c52636e3bec,34,2026-04-07 06:26:00,2026-04-07 08:17:21
2,ck_0010e31baa4a1fb7,16,2026-04-15 11:53:18,2026-04-15 23:39:34
3,ck_0010ec3874fb5378,74,2026-04-06 02:08:23,2026-04-06 17:59:27
4,ck_001722063b94cae0,15,2026-04-10 18:07:09,2026-04-10 19:15:47


In [132]:
actions['activity_per_minute'] = actions['n_events'] / (((actions['last_event'] - actions['first_event']).dt.total_seconds()/60).clip(lower=1))
actions.head()

,cookie_id,n_events,first_event,last_event,activity_per_minute
0,ck_000c95f1408dcb00,16,2026-04-19 14:49:54,2026-04-19 23:11:51,0.031876
1,ck_000e8c52636e3bec,34,2026-04-07 06:26:00,2026-04-07 08:17:21,0.305344
2,ck_0010e31baa4a1fb7,16,2026-04-15 11:53:18,2026-04-15 23:39:34,0.022654
3,ck_0010ec3874fb5378,74,2026-04-06 02:08:23,2026-04-06 17:59:27,0.077807
4,ck_001722063b94cae0,15,2026-04-10 18:07:09,2026-04-10 19:15:47,0.218553


In [133]:
features = actions.copy().drop(['first_event', 'last_event'], axis=1)
features.head()

,cookie_id,n_events,activity_per_minute
0,ck_000c95f1408dcb00,16,0.031876
1,ck_000e8c52636e3bec,34,0.305344
2,ck_0010e31baa4a1fb7,16,0.022654
3,ck_0010ec3874fb5378,74,0.077807
4,ck_001722063b94cae0,15,0.218553


In [134]:
def features_merge(features, df):
    new_df = features.merge(df, on='cookie_id', how='left')
    return new_df

### 2 признак

In [135]:
ev_tr.groupby('cookie_id').agg(
        n_events=('event_ts', 'size'),
        first_event=('event_ts', 'min'),
        last_event=('event_ts', 'max')).reset_index()

,cookie_id,n_events,first_event,last_event
0,ck_000c95f1408dcb00,16,2026-04-19 14:49:54,2026-04-19 23:11:51
1,ck_000e8c52636e3bec,34,2026-04-07 06:26:00,2026-04-07 08:17:21
2,ck_0010e31baa4a1fb7,16,2026-04-15 11:53:18,2026-04-15 23:39:34
3,ck_0010ec3874fb5378,74,2026-04-06 02:08:23,2026-04-06 17:59:27
4,ck_001722063b94cae0,15,2026-04-10 18:07:09,2026-04-10 19:15:47
...,...,...,...,...
11086,ck_ffe2bdd663cea9ac,32,2026-04-15 10:17:17,2026-04-15 15:47:29
11087,ck_ffe648839179d098,28,2026-04-12 02:19:45,2026-04-12 15:32:20
11088,ck_ffee36a37e2532f5,81,2026-04-10 11:07:35,2026-04-10 23:21:57
11089,ck_fff88a7920fd0fb6,11,2026-04-17 04:54:29,2026-04-17 06:01:22


In [136]:
events_sort = ev_tr.sort_values(['cookie_id', 'event_ts'])

In [137]:
events_sort['time_diff'] = events_sort.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()/60
events_sort.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts,time_diff
189443,ck_000c95f1408dcb00,2026-04-19 14:49:54,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,hobbi,NaN,NaN,гитара акустическая,2.0,373.0,1059.0,2026-04-19,2026-04-20,NaN
47919,ck_000c95f1408dcb00,2026-04-19 14:50:17,210,photo_swipe,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1021247.0,mebel,samara,private,NaN,NaN,1419.0,674.0,2026-04-19,2026-04-20,0.383333
143776,ck_000c95f1408dcb00,2026-04-19 15:50:21,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1021247.0,mebel,samara,private,NaN,NaN,1224.0,218.0,2026-04-19,2026-04-20,60.066667
134951,ck_000c95f1408dcb00,2026-04-19 15:50:47,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1050338.0,mebel,volgograd,private,NaN,NaN,1195.0,372.0,2026-04-19,2026-04-20,0.433333
155957,ck_000c95f1408dcb00,2026-04-19 15:51:45,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1050338.0,NaN,volgograd,private,NaN,NaN,898.0,1021.0,2026-04-19,2026-04-20,0.966667


In [138]:
time_diff = events_sort.groupby('cookie_id').agg(
    mean_time_diff=('time_diff', 'mean'),
    std_time_diff=('time_diff', 'std'),
    min_time_diff=('time_diff', 'min'),
    max_time_diff=('time_diff', 'max')).reset_index()
time_diff.head()

,cookie_id,mean_time_diff,std_time_diff,min_time_diff,max_time_diff
0,ck_000c95f1408dcb00,33.463333,51.342558,0.283333,128.883333
1,ck_000e8c52636e3bec,3.374242,12.905100,0.000000,70.950000
2,ck_0010e31baa4a1fb7,47.084444,51.795114,0.166667,137.383333
3,ck_0010ec3874fb5378,13.028311,33.073415,0.000000,142.783333
4,ck_001722063b94cae0,4.902381,7.130523,0.033333,24.116667


In [139]:
features = features_merge(features, time_diff)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff
0,ck_000c95f1408dcb00,16,0.031876,33.463333,51.342558,0.283333,128.883333
1,ck_000e8c52636e3bec,34,0.305344,3.374242,12.905100,0.000000,70.950000
2,ck_0010e31baa4a1fb7,16,0.022654,47.084444,51.795114,0.166667,137.383333
3,ck_0010ec3874fb5378,74,0.077807,13.028311,33.073415,0.000000,142.783333
4,ck_001722063b94cae0,15,0.218553,4.902381,7.130523,0.033333,24.116667


### 3 признак

здесь перешел к признакам, которые описывают сами действия

In [140]:
events_sort.groupby('cookie_id')['event_name'].agg(list)

cookie_id
ck_000c95f1408dcb00    [search_results_view, photo_swipe, item_view, ...
ck_000e8c52636e3bec    [item_view, search_results_view, favorite_add,...
ck_0010e31baa4a1fb7    [item_view, search_results_view, item_view, fa...
ck_0010ec3874fb5378    [favorite_add, favorite_add, item_view, search...
ck_001722063b94cae0    [item_view, search_results_view, item_view, it...
                                             ...                        
ck_ffe2bdd663cea9ac    [photo_swipe, favorite_add, contact_phone_show...
ck_ffe648839179d098    [search_results_view, favorite_add, item_view,...
ck_ffee36a37e2532f5    [seller_page_view, search_results_view, search...
ck_fff88a7920fd0fb6    [item_view, item_view, item_view, photo_swipe,...
ck_fffa4b3ab0203fcf    [item_view, item_view, search_results_view, it...
Name: event_name, Length: 11091, dtype: object

In [141]:
events_sort['previous_event'] = events_sort.groupby('cookie_id')['event_name'].shift(1)
events_sort.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts,time_diff,previous_event
189443,ck_000c95f1408dcb00,2026-04-19 14:49:54,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,hobbi,NaN,NaN,гитара акустическая,2.0,373.0,1059.0,2026-04-19,2026-04-20,NaN,NaN
47919,ck_000c95f1408dcb00,2026-04-19 14:50:17,210,photo_swipe,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1021247.0,mebel,samara,private,NaN,NaN,1419.0,674.0,2026-04-19,2026-04-20,0.383333,search_results_view
143776,ck_000c95f1408dcb00,2026-04-19 15:50:21,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1021247.0,mebel,samara,private,NaN,NaN,1224.0,218.0,2026-04-19,2026-04-20,60.066667,photo_swipe
134951,ck_000c95f1408dcb00,2026-04-19 15:50:47,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1050338.0,mebel,volgograd,private,NaN,NaN,1195.0,372.0,2026-04-19,2026-04-20,0.433333,item_view
155957,ck_000c95f1408dcb00,2026-04-19 15:51:45,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1050338.0,NaN,volgograd,private,NaN,NaN,898.0,1021.0,2026-04-19,2026-04-20,0.966667,item_view


In [142]:
events_sort['same_as_previous'] = (events_sort['event_name'] == events_sort['previous_event']).where(events_sort['previous_event'].notna())

repeat_features = events_sort.groupby('cookie_id').agg(
    repeated_actions=('same_as_previous', 'sum'),
    repeated_actions_ratio=('same_as_previous', 'mean')).reset_index()

repeat_features.head()

,cookie_id,repeated_actions,repeated_actions_ratio
0,ck_000c95f1408dcb00,7,0.466667
1,ck_000e8c52636e3bec,5,0.151515
2,ck_0010e31baa4a1fb7,3,0.2
3,ck_0010ec3874fb5378,21,0.287671
4,ck_001722063b94cae0,4,0.285714


In [143]:
features = features_merge(features, repeat_features)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,repeated_actions_ratio
0,ck_000c95f1408dcb00,16,0.031876,33.463333,51.342558,0.283333,128.883333,7,0.466667
1,ck_000e8c52636e3bec,34,0.305344,3.374242,12.905100,0.000000,70.950000,5,0.151515
2,ck_0010e31baa4a1fb7,16,0.022654,47.084444,51.795114,0.166667,137.383333,3,0.2
3,ck_0010ec3874fb5378,74,0.077807,13.028311,33.073415,0.000000,142.783333,21,0.287671
4,ck_001722063b94cae0,15,0.218553,4.902381,7.130523,0.033333,24.116667,4,0.285714


### 4 признак

In [144]:
proportions = ev_tr.groupby('cookie_id')['event_name'].value_counts(normalize=True).reset_index()
proportions.head()

,cookie_id,event_name,proportion
0,ck_000c95f1408dcb00,item_view,0.687500
1,ck_000c95f1408dcb00,search_results_view,0.187500
2,ck_000c95f1408dcb00,photo_swipe,0.062500
3,ck_000c95f1408dcb00,seller_page_view,0.062500
4,ck_000e8c52636e3bec,item_view,0.323529


In [145]:
proportions['entropy_part'] = -proportions['proportion'] * np.log(proportions['proportion'])
proportions.head()

,cookie_id,event_name,proportion,entropy_part
0,ck_000c95f1408dcb00,item_view,0.687500,0.257602
1,ck_000c95f1408dcb00,search_results_view,0.187500,0.313871
2,ck_000c95f1408dcb00,photo_swipe,0.062500,0.173287
3,ck_000c95f1408dcb00,seller_page_view,0.062500,0.173287
4,ck_000e8c52636e3bec,item_view,0.323529,0.365092


In [146]:
event_entropy = proportions.groupby('cookie_id')['entropy_part'].sum().rename('event_entropy').reset_index()
event_entropy.head()


,cookie_id,event_entropy
0,ck_000c95f1408dcb00,0.918046
1,ck_000e8c52636e3bec,1.672241
2,ck_0010e31baa4a1fb7,1.488036
3,ck_0010ec3874fb5378,1.521574
4,ck_001722063b94cae0,1.062535


In [147]:
features = features_merge(features, event_entropy)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,repeated_actions_ratio,event_entropy
0,ck_000c95f1408dcb00,16,0.031876,33.463333,51.342558,0.283333,128.883333,7,0.466667,0.918046
1,ck_000e8c52636e3bec,34,0.305344,3.374242,12.905100,0.000000,70.950000,5,0.151515,1.672241
2,ck_0010e31baa4a1fb7,16,0.022654,47.084444,51.795114,0.166667,137.383333,3,0.2,1.488036
3,ck_0010ec3874fb5378,74,0.077807,13.028311,33.073415,0.000000,142.783333,21,0.287671,1.521574
4,ck_001722063b94cae0,15,0.218553,4.902381,7.130523,0.033333,24.116667,4,0.285714,1.062535


### 5 признак

In [148]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


In [149]:
pointer = ev_tr.groupby('cookie_id').agg(
    pointer_count=('pointer_x', 'count'),
    pointer_x_nunique=('pointer_x', 'nunique'),
    pointer_y_nunique=('pointer_y', 'nunique'),
    pointer_x_std=('pointer_x', 'std'),
    pointer_y_std=('pointer_y', 'std'),
    pointer_x_min=('pointer_x', 'min'),
    pointer_x_max=('pointer_x', 'max'),
    pointer_y_min=('pointer_y', 'min'),
    pointer_y_max=('pointer_y', 'max')).reset_index()

pointer.head()

,cookie_id,pointer_count,pointer_x_nunique,pointer_y_nunique,pointer_x_std,pointer_y_std,pointer_x_min,pointer_x_max,pointer_y_min,pointer_y_max
0,ck_000c95f1408dcb00,16,16,16,494.980875,338.074640,2.0,1884.0,106.0,1059.0
1,ck_000e8c52636e3bec,31,31,31,561.544146,277.719426,18.0,1910.0,28.0,1048.0
2,ck_0010e31baa4a1fb7,16,16,15,501.890941,361.434157,72.0,1522.0,2.0,1023.0
3,ck_0010ec3874fb5378,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,ck_001722063b94cae0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [150]:
features = features_merge(features, pointer)
features.head()

,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,repeated_actions_ratio,event_entropy,pointer_count,pointer_x_nunique,pointer_y_nunique,pointer_x_std,pointer_y_std,pointer_x_min,pointer_x_max,pointer_y_min,pointer_y_max
0,ck_000c95f1408dcb00,16,0.031876,33.463333,51.342558,0.283333,128.883333,7,0.466667,0.918046,16,16,16,494.980875,338.074640,2.0,1884.0,106.0,1059.0
1,ck_000e8c52636e3bec,34,0.305344,3.374242,12.905100,0.000000,70.950000,5,0.151515,1.672241,31,31,31,561.544146,277.719426,18.0,1910.0,28.0,1048.0
2,ck_0010e31baa4a1fb7,16,0.022654,47.084444,51.795114,0.166667,137.383333,3,0.2,1.488036,16,16,15,501.890941,361.434157,72.0,1522.0,2.0,1023.0
3,ck_0010ec3874fb5378,74,0.077807,13.028311,33.073415,0.000000,142.783333,21,0.287671,1.521574,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,ck_001722063b94cae0,15,0.218553,4.902381,7.130523,0.033333,24.116667,4,0.285714,1.062535,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## Разделение на train и validation

Так как тестовые данные идут после обучающих по времени, для валидации используем последние дни train. Это лучше имитирует ситуацию, в которой модель предсказывает поведение будущих кук.

In [151]:
model_data = train[['cookie_id', 'window_start_ts', 'target']].merge(features, on='cookie_id', how='left', validate='one_to_one')

print('Размер model_data:', model_data.shape)
print('Уникальных cookie_id:', model_data['cookie_id'].nunique())
model_data.head()

Размер model_data: (11091, 21)
Уникальных cookie_id: 11091


,cookie_id,window_start_ts,target,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,...,event_entropy,pointer_count,pointer_x_nunique,pointer_y_nunique,pointer_x_std,pointer_y_std,pointer_x_min,pointer_x_max,pointer_y_min,pointer_y_max
0,ck_54a059eb7d3ea68b,2026-04-06,0,7,2.514970,0.463889,0.499157,0.016667,1.400000,2,...,1.004242,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,ck_7e4de46eeab82974,2026-04-06,0,41,0.062316,16.448333,40.271448,0.033333,141.850000,17,...,1.378723,38,36,38,509.231969,272.953569,2.0,1907.0,104.0,1053.0
2,ck_9320229ef6304522,2026-04-06,0,36,0.059193,17.376667,38.440823,0.016667,139.166667,5,...,1.874204,35,35,35,602.124295,312.027426,35.0,1904.0,22.0,1057.0
3,ck_30ccd25bc1714ed9,2026-04-06,0,21,0.073166,14.350833,41.701358,0.100000,143.400000,9,...,1.062192,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,ck_a77c5f05948cdeef,2026-04-06,0,29,0.091109,11.367857,28.682575,0.000000,120.400000,8,...,1.494116,28,27,27,450.787816,297.263925,30.0,1791.0,5.0,1041.0


In [152]:
is_valid = model_data.window_start_ts.ge('2026-04-17').values

print('Даты train:', model_data.loc[~is_valid, 'window_start_ts'].min(), '-', model_data.loc[~is_valid, 'window_start_ts'].max())
print('Даты validation:', model_data.loc[is_valid, 'window_start_ts'].min(), '-', model_data.loc[is_valid, 'window_start_ts'].max())

Даты train: 2026-04-06 00:00:00 - 2026-04-16 00:00:00
Даты validation: 2026-04-17 00:00:00 - 2026-04-19 00:00:00


In [153]:
cols = [column for column in features.columns if column != 'cookie_id']

y = model_data.target.values

X_train = model_data.loc[~is_valid, cols].reset_index(drop=True)
y_train = y[~is_valid]

X_valid = model_data.loc[is_valid, cols].reset_index(drop=True)
y_valid = y[is_valid]

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('Доля ботов в train:', y_train.mean().round(4))
print('Доля ботов в validation:', y_valid.mean().round(4))

X_train: (9140, 18)
X_valid: (1951, 18)
Доля ботов в train: 0.0809
Доля ботов в validation: 0.082


## Baseline и оценка на validation

In [154]:
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall

In [155]:
model = CatBoostClassifier(
    iterations=700,
    depth=7,
    learning_rate=0.04,
    loss_function='Logloss',
    eval_metric='PRAUC',
    l2_leaf_reg=5,
    random_seed=42,
    allow_writing_files=False,
    verbose=100)

model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100)

0:	learn: 0.3643127	test: 0.2969578	best: 0.2969578 (0)	total: 2.78ms	remaining: 1.95s
100:	learn: 0.6882088	test: 0.5535767	best: 0.5557130 (95)	total: 268ms	remaining: 1.59s
200:	learn: 0.7352521	test: 0.5696048	best: 0.5698112 (199)	total: 537ms	remaining: 1.33s
300:	learn: 0.7779205	test: 0.5731124	best: 0.5742734 (275)	total: 813ms	remaining: 1.08s
400:	learn: 0.8110788	test: 0.5748978	best: 0.5754329 (399)	total: 1.14s	remaining: 852ms
500:	learn: 0.8432386	test: 0.5760716	best: 0.5765796 (442)	total: 1.4s	remaining: 558ms
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.5765796497
bestIteration = 442

Shrink model to first 443 iterations.


CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='PRAUC', iterations=700, l2_leaf_reg=5, learning_rate=0.04, loss_function='Logloss', random_seed=42, verbose=100)

In [156]:
p_valid = model.predict_proba(X_valid)[:, 1]

print('Лучшая итерация:', model.get_best_iteration())
print('P@R>=0.70:', round(precision_at_recall(y_valid, p_valid), 4))
print('PR-AUC:', round(average_precision_score(y_valid, p_valid), 4))
print('ROC-AUC:', round(roc_auc_score(y_valid, p_valid), 4))

Лучшая итерация: 442
P@R>=0.70: 0.3556
PR-AUC: 0.5776
ROC-AUC: 0.8708


## Дополнительные категориальные и агрегированные признаки

Для каждого cookie_id добавим количества и доли типов событий и платформ. Также посчитаем количество уникальных значений в категориальных полях

In [ ]:
event_counts = pd.crosstab(
    ev_tr['cookie_id'],
    ev_tr['event_name'])

event_counts.columns = ['event_count_' + column for column in event_counts.columns]
event_counts = event_counts.reset_index()

event_ratios = pd.crosstab(ev_tr['cookie_id'], ev_tr['event_name'], normalize='index')
event_ratios.columns = ['event_ratio_' + column for column in event_ratios.columns]
event_ratios = event_ratios.reset_index()

event_counts.head()

,cookie_id,event_count_contact_chat_open,event_count_contact_message_sent,event_count_contact_phone_show,event_count_favorite_add,event_count_item_view,event_count_login,event_count_photo_swipe,event_count_search_results_view,event_count_seller_page_view
0,ck_000c95f1408dcb00,0,0,0,0,11,0,1,3,1
1,ck_000e8c52636e3bec,0,0,1,4,11,1,5,9,3
2,ck_0010e31baa4a1fb7,0,0,0,2,7,1,1,4,1
3,ck_0010ec3874fb5378,0,0,0,10,25,1,14,20,4
4,ck_001722063b94cae0,0,0,1,0,8,0,1,5,0


In [ ]:
platform_counts = pd.crosstab(ev_tr['cookie_id'], ev_tr['platform'])
platform_counts.columns = ['platform_count_' + column for column in platform_counts.columns]
platform_counts = platform_counts.reset_index()

platform_ratios = pd.crosstab(ev_tr['cookie_id'], ev_tr['platform'], normalize='index')
platform_ratios.columns = ['platform_ratio_' + column for column in platform_ratios.columns]
platform_ratios = platform_ratios.reset_index()

platform_counts.head()

,cookie_id,platform_count_android,platform_count_desktop,platform_count_ios,platform_count_web
0,ck_000c95f1408dcb00,0,1,0,15
1,ck_000e8c52636e3bec,0,14,0,20
2,ck_0010e31baa4a1fb7,0,5,0,11
3,ck_0010ec3874fb5378,0,17,0,57
4,ck_001722063b94cae0,15,0,0,0


In [ ]:
unique_features = ev_tr.groupby('cookie_id').agg(
    event_type_nunique=('event_name', 'nunique'),
    platform_nunique=('platform', 'nunique'),
    user_agent_nunique=('user_agent', 'nunique'),
    item_nunique=('item_id', 'nunique'),
    item_category_nunique=('item_category', 'nunique'),
    item_location_nunique=('item_location', 'nunique'),
    seller_type_nunique=('seller_type', 'nunique'),
    search_query_nunique=('search_query', 'nunique'),
    search_page_nunique=('search_page', 'nunique')).reset_index()

unique_features.head()

,cookie_id,event_type_nunique,platform_nunique,user_agent_nunique,item_nunique,item_category_nunique,item_location_nunique,seller_type_nunique,search_query_nunique,search_page_nunique
0,ck_000c95f1408dcb00,4,2,1,9,2,8,2,3,3
1,ck_000e8c52636e3bec,7,2,1,20,4,12,2,8,6
2,ck_0010e31baa4a1fb7,6,2,1,8,3,4,2,4,1
3,ck_0010ec3874fb5378,6,2,1,31,2,19,2,8,4
4,ck_001722063b94cae0,4,1,1,8,3,9,2,4,2


In [ ]:
features_improved = features.copy()

for new_features in [event_counts, event_ratios, platform_counts, platform_ratios, unique_features]:
    features_improved = features_merge(features_improved, new_features)

print('Признаков в baseline:', len(features.columns) - 1)
print('Признаков после улучшения:', len(features_improved.columns) - 1)
features_improved.head()

Признаков в baseline: 18
Признаков после улучшения: 53


,cookie_id,n_events,activity_per_minute,mean_time_diff,std_time_diff,min_time_diff,max_time_diff,repeated_actions,repeated_actions_ratio,event_entropy,...,platform_ratio_web,event_type_nunique,platform_nunique,user_agent_nunique,item_nunique,item_category_nunique,item_location_nunique,seller_type_nunique,search_query_nunique,search_page_nunique
0,ck_000c95f1408dcb00,16,0.031876,33.463333,51.342558,0.283333,128.883333,7,0.466667,0.918046,...,0.937500,4,2,1,9,2,8,2,3,3
1,ck_000e8c52636e3bec,34,0.305344,3.374242,12.905100,0.000000,70.950000,5,0.151515,1.672241,...,0.588235,7,2,1,20,4,12,2,8,6
2,ck_0010e31baa4a1fb7,16,0.022654,47.084444,51.795114,0.166667,137.383333,3,0.2,1.488036,...,0.687500,6,2,1,8,3,4,2,4,1
3,ck_0010ec3874fb5378,74,0.077807,13.028311,33.073415,0.000000,142.783333,21,0.287671,1.521574,...,0.770270,6,2,1,31,2,19,2,8,4
4,ck_001722063b94cae0,15,0.218553,4.902381,7.130523,0.033333,24.116667,4,0.285714,1.062535,...,0.000000,4,1,1,8,3,9,2,4,2


## Сравнение baseline и модели с новыми признаками

Используем ту же временную границу и те же параметры CatBoost. Поэтому разница в метриках будет связана именно с добавленными признаками.

In [ ]:
model_data_improved = train[['cookie_id', 'window_start_ts', 'target']].merge(
    features_improved,
    on='cookie_id',
    how='left',
    validate='one_to_one')

cols_improved = [column for column in features_improved.columns if column != 'cookie_id']

X_train_improved = model_data_improved.loc[~is_valid, cols_improved].reset_index(drop=True)
X_valid_improved = model_data_improved.loc[is_valid, cols_improved].reset_index(drop=True)

print('X_train_improved:', X_train_improved.shape)
print('X_valid_improved:', X_valid_improved.shape)

X_train_improved: (9140, 53)
X_valid_improved: (1951, 53)


In [ ]:
model_improved = CatBoostClassifier(
    iterations=700,
    depth=7,
    learning_rate=0.04,
    loss_function='Logloss',
    eval_metric='PRAUC',
    l2_leaf_reg=5,
    random_seed=42,
    allow_writing_files=False,
    verbose=100)

model_improved.fit(
    X_train_improved,
    y_train,
    eval_set=(X_valid_improved, y_valid),
    early_stopping_rounds=100)

0:	learn: 0.4436988	test: 0.3790946	best: 0.3790946 (0)	total: 4.54ms	remaining: 3.17s
100:	learn: 0.7852588	test: 0.6750761	best: 0.6750761 (100)	total: 331ms	remaining: 1.97s
200:	learn: 0.8451929	test: 0.6903174	best: 0.6908189 (199)	total: 650ms	remaining: 1.61s
300:	learn: 0.8867653	test: 0.7006012	best: 0.7006012 (300)	total: 970ms	remaining: 1.29s
400:	learn: 0.9121270	test: 0.7032143	best: 0.7036263 (398)	total: 1.29s	remaining: 962ms
500:	learn: 0.9350224	test: 0.7084950	best: 0.7087176 (497)	total: 1.61s	remaining: 640ms
600:	learn: 0.9526075	test: 0.7101831	best: 0.7110086 (586)	total: 1.94s	remaining: 319ms
699:	learn: 0.9671173	test: 0.7105854	best: 0.7121198 (608)	total: 2.25s	remaining: 0us

bestTest = 0.712119803
bestIteration = 608

Shrink model to first 609 iterations.


CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='PRAUC', iterations=700, l2_leaf_reg=5, learning_rate=0.04, loss_function='Logloss', random_seed=42, verbose=100)

In [ ]:
p_valid_improved = model_improved.predict_proba(X_valid_improved)[:, 1]

comparison = pd.DataFrame({
    'model': ['baseline', 'improved'],
    'n_features': [len(cols), len(cols_improved)],
    'P@R>=0.70': [precision_at_recall(y_valid, p_valid), precision_at_recall(y_valid, p_valid_improved)],
    'PR-AUC': [average_precision_score(y_valid, p_valid), average_precision_score(y_valid, p_valid_improved)],
    'ROC-AUC': [roc_auc_score(y_valid, p_valid), roc_auc_score(y_valid, p_valid_improved)]})

comparison.round(4)

,model,n_features,P@R>=0.70,PR-AUC,ROC-AUC
0,baseline,18,0.3556,0.5776,0.8708
1,improved,53,0.5437,0.7128,0.9144


## Эксперимент с дубликатами

Изначально я думал провести эксперимент, в котором удалю дубликаты и посмотрю на зименение метрики. Но так как я выполняю задание в концу дедлайна, решил не проводить эксперимент из-за того, что много подготовки нужно сделать. Вместо этого я решил добавить новые признаки связанные с дубликатами и провести оценку

In [ ]:
event_columns = events.columns.tolist()

events_duplicates = ev_tr.copy()

events_duplicates['is_duplicated'] = events_duplicates.duplicated(subset=event_columns, keep='first')

In [ ]:
duplicate_features = events_duplicates.groupby('cookie_id').agg(
    duplicate_count=('is_duplicated', 'sum'),
    duplicate_ratio=('is_duplicated', 'mean'),
    has_duplicates=('is_duplicated', 'max')).reset_index()

In [ ]:
features_improved_dup = features_merge(
    features_improved,
    duplicate_features)

print('Признаков до добавления дубликатов:', len(features_improved.columns) - 1)
print('Признаков после добавления дубликатов:', len(features_improved_dup.columns) - 1)
features_improved_dup[['cookie_id', 'duplicate_count', 'duplicate_ratio', 'has_duplicates']].head()

Признаков до добавления дубликатов: 53
Признаков после добавления дубликатов: 56


,cookie_id,duplicate_count,duplicate_ratio,has_duplicates
0,ck_000c95f1408dcb00,0,0.0,False
1,ck_000e8c52636e3bec,0,0.0,False
2,ck_0010e31baa4a1fb7,0,0.0,False
3,ck_0010ec3874fb5378,0,0.0,False
4,ck_001722063b94cae0,0,0.0,False


## Оценка новых признаков

In [ ]:
model_data_improved_dup = train[['cookie_id', 'window_start_ts', 'target']].merge(
    features_improved_dup,
    on='cookie_id',
    how='left',
    validate='one_to_one')

cols_improved_dup = [column for column in features_improved_dup.columns if column != 'cookie_id']

X_train_improved_dup = model_data_improved_dup.loc[~is_valid, cols_improved_dup].reset_index(drop=True)

X_valid_improved_dup = model_data_improved_dup.loc[is_valid, cols_improved_dup].reset_index(drop=True)

print('X_train_improved_dup:', X_train_improved_dup.shape)
print('X_valid_improved_dup:', X_valid_improved_dup.shape)

X_train_improved_dup: (9140, 56)
X_valid_improved_dup: (1951, 56)


In [ ]:
model_improved_dup = CatBoostClassifier(
    iterations=700,
    depth=7,
    learning_rate=0.04,
    loss_function='Logloss',
    eval_metric='PRAUC',
    l2_leaf_reg=5,
    random_seed=42,
    allow_writing_files=False,
    verbose=100)

model_improved_dup.fit(
    X_train_improved_dup,
    y_train,
    eval_set=(X_valid_improved_dup, y_valid),
    early_stopping_rounds=100)

0:	learn: 0.3394567	test: 0.2842959	best: 0.2842959 (0)	total: 3.64ms	remaining: 2.54s
100:	learn: 0.7880505	test: 0.6693861	best: 0.6704292 (98)	total: 336ms	remaining: 1.99s
200:	learn: 0.8425594	test: 0.6861276	best: 0.6861276 (200)	total: 665ms	remaining: 1.65s
300:	learn: 0.8845474	test: 0.6967612	best: 0.6967612 (300)	total: 995ms	remaining: 1.32s
400:	learn: 0.9080935	test: 0.7012310	best: 0.7012611 (396)	total: 1.32s	remaining: 987ms
500:	learn: 0.9275036	test: 0.7024184	best: 0.7030824 (496)	total: 1.65s	remaining: 657ms
600:	learn: 0.9435072	test: 0.7031860	best: 0.7043471 (573)	total: 1.98s	remaining: 327ms
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7043470886
bestIteration = 573

Shrink model to first 574 iterations.


CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='PRAUC', iterations=700, l2_leaf_reg=5, learning_rate=0.04, loss_function='Logloss', random_seed=42, verbose=100)

In [ ]:
p_valid_improved_dup = model_improved_dup.predict_proba(X_valid_improved_dup)[:, 1]

new_result = {
    'model': 'improved_dup',
    'n_features': len(cols_improved_dup),
    'P@R>=0.70': precision_at_recall(y_valid, p_valid_improved_dup),
    'PR-AUC': average_precision_score(y_valid, p_valid_improved_dup),
    'ROC-AUC': roc_auc_score(y_valid, p_valid_improved_dup)}

comparison = comparison[comparison['model'] != new_result['model']].copy()

comparison.loc[len(comparison)] = new_result
comparison.round(4)

,model,n_features,P@R>=0.70,PR-AUC,ROC-AUC
0,baseline,18,0.3556,0.5776,0.8708
1,improved,53,0.5437,0.7128,0.9144
2,improved_dup,56,0.4873,0.7050,0.9127


## Подготовка test и создание submission

Для ev_te строим те же признаки, что и для улучшенной обучающей выборки

In [170]:
actions_te = ev_te.groupby('cookie_id').agg(
    n_events=('event_ts', 'size'),
    first_event=('event_ts', 'min'),
    last_event=('event_ts', 'max')).reset_index()

activity_minutes_te = (actions_te['last_event'] - actions_te['first_event']).dt.total_seconds()/60

actions_te['activity_per_minute'] = (actions_te['n_events'] / activity_minutes_te.clip(lower=1))

features_te = test[['cookie_id']].copy()
features_te = features_merge(features_te,actions_te.drop(['first_event', 'last_event'], axis=1))

features_te.head()

,cookie_id,n_events,activity_per_minute
0,ck_315fb710a0e371e7,80,0.109302
1,ck_a76ee3b3e3e522fd,9,2.857143
2,ck_94c9a4d382689e82,8,0.066537
3,ck_8eaf9509ad9462a0,9,0.945709
4,ck_9a88a5a989cb5bc6,2,1.500000


In [171]:
events_sort_te = ev_te.sort_values(['cookie_id', 'event_ts']).copy()

events_sort_te['time_diff'] = (events_sort_te.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()/60)

time_diff_te = events_sort_te.groupby('cookie_id').agg(
    mean_time_diff=('time_diff', 'mean'),
    std_time_diff=('time_diff', 'std'),
    min_time_diff=('time_diff', 'min'),
    max_time_diff=('time_diff', 'max')).reset_index()

features_te = features_merge(features_te, time_diff_te)
time_diff_te.head()

,cookie_id,mean_time_diff,std_time_diff,min_time_diff,max_time_diff
0,ck_00013fdfa0bd37dd,9.107143,15.610213,0.016667,48.000000
1,ck_001e42551cd06e40,29.214286,52.800072,0.116667,138.200000
2,ck_0027b5d4107d65d8,24.153333,43.540212,0.000000,129.133333
3,ck_0027ba644f9ae2cd,12.130682,32.053346,0.000000,140.450000
4,ck_002cf760d173f0f8,1.287500,1.018021,0.283333,2.616667


In [172]:
events_sort_te['previous_event'] = (events_sort_te.groupby('cookie_id')['event_name'].shift(1))

events_sort_te['same_as_previous'] = (events_sort_te['event_name'] == events_sort_te['previous_event']).where(events_sort_te['previous_event'].notna())

repeat_features_te = events_sort_te.groupby('cookie_id').agg(
    repeated_actions=('same_as_previous', 'sum'),
    repeated_actions_ratio=('same_as_previous', 'mean')).reset_index()

features_te = features_merge(features_te, repeat_features_te)
repeat_features_te.head()

,cookie_id,repeated_actions,repeated_actions_ratio
0,ck_00013fdfa0bd37dd,1,0.071429
1,ck_001e42551cd06e40,0,0.0
2,ck_0027b5d4107d65d8,7,0.466667
3,ck_0027ba644f9ae2cd,13,0.295455
4,ck_002cf760d173f0f8,0,0.0


In [173]:
proportions_te = (
    ev_te.groupby('cookie_id')['event_name']
    .value_counts(normalize=True)
    .rename('proportion')
    .reset_index())

proportions_te['entropy_part'] = (
    -proportions_te['proportion']
    * np.log(proportions_te['proportion']))

event_entropy_te = (
    proportions_te.groupby('cookie_id')['entropy_part']
    .sum()
    .rename('event_entropy')
    .reset_index())

features_te = features_merge(features_te, event_entropy_te)
event_entropy_te.head()

,cookie_id,event_entropy
0,ck_00013fdfa0bd37dd,1.714088
1,ck_001e42551cd06e40,1.494175
2,ck_0027b5d4107d65d8,1.511086
3,ck_0027ba644f9ae2cd,1.314161
4,ck_002cf760d173f0f8,0.673012


In [174]:
pointer_te = ev_te.groupby('cookie_id').agg(
    pointer_count=('pointer_x', 'count'),
    pointer_x_nunique=('pointer_x', 'nunique'),
    pointer_y_nunique=('pointer_y', 'nunique'),
    pointer_x_std=('pointer_x', 'std'),
    pointer_y_std=('pointer_y', 'std'),
    pointer_x_min=('pointer_x', 'min'),
    pointer_x_max=('pointer_x', 'max'),
    pointer_y_min=('pointer_y', 'min'),
    pointer_y_max=('pointer_y', 'max')).reset_index()

features_te = features_merge(features_te, pointer_te)
pointer_te.head()

,cookie_id,pointer_count,pointer_x_nunique,pointer_y_nunique,pointer_x_std,pointer_y_std,pointer_x_min,pointer_x_max,pointer_y_min,pointer_y_max
0,ck_00013fdfa0bd37dd,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,ck_001e42551cd06e40,7,7,7,440.479609,334.232668,264.0,1399.0,13.0,1078.0
2,ck_0027b5d4107d65d8,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,ck_0027ba644f9ae2cd,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,ck_002cf760d173f0f8,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [175]:
event_counts_te = pd.crosstab(ev_te['cookie_id'], ev_te['event_name'])
event_counts_te.columns = ['event_count_' + column for column in event_counts_te.columns]
event_counts_te = event_counts_te.reset_index()

event_ratios_te = pd.crosstab(
    ev_te['cookie_id'],
    ev_te['event_name'],
    normalize='index')
event_ratios_te.columns = ['event_ratio_' + column for column in event_ratios_te.columns]
event_ratios_te = event_ratios_te.reset_index()

features_te = features_merge(features_te, event_counts_te)
features_te = features_merge(features_te, event_ratios_te)

event_counts_te.head()

,cookie_id,event_count_contact_chat_open,event_count_contact_message_sent,event_count_contact_phone_show,event_count_favorite_add,event_count_item_view,event_count_login,event_count_photo_swipe,event_count_search_results_view,event_count_seller_page_view
0,ck_00013fdfa0bd37dd,2,0,0,1,4,2,3,3,0
1,ck_001e42551cd06e40,0,0,0,1,3,1,2,1,0
2,ck_0027b5d4107d65d8,0,0,3,0,5,0,3,4,1
3,ck_0027ba644f9ae2cd,0,0,4,2,23,0,3,12,1
4,ck_002cf760d173f0f8,0,0,0,0,2,0,0,3,0


In [176]:
platform_counts_te = pd.crosstab(ev_te['cookie_id'], ev_te['platform'])
platform_counts_te.columns = ['platform_count_' + column for column in platform_counts_te.columns]
platform_counts_te = platform_counts_te.reset_index()

platform_ratios_te = pd.crosstab(
    ev_te['cookie_id'],
    ev_te['platform'],
    normalize='index')
platform_ratios_te.columns = ['platform_ratio_' + column for column in platform_ratios_te.columns]
platform_ratios_te = platform_ratios_te.reset_index()

features_te = features_merge(features_te, platform_counts_te)
features_te = features_merge(features_te, platform_ratios_te)

platform_counts_te.head()

,cookie_id,platform_count_android,platform_count_desktop,platform_count_ios,platform_count_web
0,ck_00013fdfa0bd37dd,15,0,0,0
1,ck_001e42551cd06e40,0,3,0,5
2,ck_0027b5d4107d65d8,16,0,0,0
3,ck_0027ba644f9ae2cd,45,0,0,0
4,ck_002cf760d173f0f8,0,0,0,5


In [177]:
unique_features_te = ev_te.groupby('cookie_id').agg(
    event_type_nunique=('event_name', 'nunique'),
    platform_nunique=('platform', 'nunique'),
    user_agent_nunique=('user_agent', 'nunique'),
    item_nunique=('item_id', 'nunique'),
    item_category_nunique=('item_category', 'nunique'),
    item_location_nunique=('item_location', 'nunique'),
    seller_type_nunique=('seller_type', 'nunique'),
    search_query_nunique=('search_query', 'nunique'),
    search_page_nunique=('search_page', 'nunique')).reset_index()

features_te = features_merge(features_te, unique_features_te)
unique_features_te.head()

,cookie_id,event_type_nunique,platform_nunique,user_agent_nunique,item_nunique,item_category_nunique,item_location_nunique,seller_type_nunique,search_query_nunique,search_page_nunique
0,ck_00013fdfa0bd37dd,6,1,1,9,3,4,2,3,3
1,ck_001e42551cd06e40,5,2,3,4,2,3,1,1,1
2,ck_0027b5d4107d65d8,5,1,1,8,1,8,2,3,2
3,ck_0027ba644f9ae2cd,6,1,1,26,2,18,2,6,5
4,ck_002cf760d173f0f8,2,1,1,2,1,2,2,3,2


In [178]:
features_te = features_te.reindex(columns=features_improved.columns, fill_value=0)

print('Train features:', features_improved.shape)
print('Test features:', features_te.shape)
print('Порядок колонок совпадает:', list(features_te.columns) == list(features_improved.columns))

Train features: (11091, 54)
Test features: (4909, 54)
Порядок колонок совпадает: True


## Финальная модель

Количество итераций берём из лучшей итерации улучшенной модели на временной валидации. Затем обучаем CatBoost на всех обучающих даных

In [179]:
final_train_data = train[['cookie_id', 'target']].merge(
    features_improved,
    on='cookie_id',
    how='left',
    validate='one_to_one')

final_test_data = test[['cookie_id']].merge(
    features_te,
    on='cookie_id',
    how='left',
    validate='one_to_one')

X_all = final_train_data[cols_improved]
y_all = final_train_data['target']
X_test = final_test_data[cols_improved]

best_iteration = model_improved.get_best_iteration()
final_iterations = best_iteration + 1 if best_iteration >= 0 else 700

print('X_all:', X_all.shape)
print('X_test:', X_test.shape)
print('Итераций финальной модели:', final_iterations)

X_all: (11091, 53)
X_test: (4909, 53)
Итераций финальной модели: 609


In [180]:
final_model = CatBoostClassifier(
    iterations=final_iterations,
    depth=7,
    learning_rate=0.04,
    loss_function='Logloss',
    l2_leaf_reg=5,
    random_seed=42,
    allow_writing_files=False,
    verbose=100)

final_model.fit(X_all, y_all)

0:	learn: 0.6313216	total: 10.5ms	remaining: 6.39s
100:	learn: 0.1384042	total: 364ms	remaining: 1.83s
200:	learn: 0.1167904	total: 689ms	remaining: 1.4s
300:	learn: 0.1020117	total: 1.02s	remaining: 1.04s
400:	learn: 0.0909744	total: 1.34s	remaining: 696ms
500:	learn: 0.0823482	total: 1.68s	remaining: 361ms
600:	learn: 0.0751071	total: 2s	remaining: 26.7ms
608:	learn: 0.0745436	total: 2.03s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, depth=7, iterations=609, l2_leaf_reg=5, learning_rate=0.04, loss_function='Logloss', random_seed=42, verbose=100)

In [181]:
test_score = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({'cookie_id': final_test_data['cookie_id'], 'score': test_score})

submission.to_csv('submission.csv', index=False)
print('Сохранён submission.csv, строк:', len(submission))
submission.head()

Сохранён submission.csv, строк: 4909


,cookie_id,score
0,ck_315fb710a0e371e7,0.000580
1,ck_a76ee3b3e3e522fd,0.082541
2,ck_94c9a4d382689e82,0.013075
3,ck_8eaf9509ad9462a0,0.000901
4,ck_9a88a5a989cb5bc6,0.017211
